# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"Dataset: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Version: {metadata['version']}")
print(f"Published: {metadata['datePublished']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their field @ids
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} Record Sets:")
all_fields = dict()  # Map of record_set_id → fields (list of (id, name))
for rs in record_sets:
    rsm = rs.to_json() if hasattr(rs, 'to_json') else rs
    print(f"- Record Set @id: {rsm['@id']}, name: {rsm.get('name', '')}")
    fields = rsm.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    elif not isinstance(fields, list):
        fields = []
    field_list = []
    print(f"  Fields:")
    for field in fields:
        if isinstance(field, dict):
            fid = field.get('@id', str(field))
            fname = field.get('name', '')
        else:
            fid = str(field)
            fname = ''
        print(f"   - @id: {fid}, name: {fname}")
        field_list.append((fid, fname))
    all_fields[rsm['@id']] = field_list

# Preview records from first record set, using its @id
if len(record_sets) > 0:
    first_rs_id = record_sets[0]['@id'] if isinstance(record_sets[0], dict) else record_sets[0].to_json()['@id']
    print(f"\nSample records from {first_rs_id}:")
    try:
        for i, rec in enumerate(dataset.records(record_set=first_rs_id)):
            print(json.dumps(rec, indent=2))
            if i >= 2: break
    except Exception as e:
        print(f"Error retrieving records: {e}")

## 3. Data Extraction
Load data from the record sets into DataFrames for analysis. Use the `@id` values for each entity.

In [ ]:
# Extract data from all available record sets
df_map = dict()  # Map: record_set @id → DataFrame
for rs in dataset.metadata.record_sets:
    rs_id = rs['@id'] if isinstance(rs, dict) else rs.to_json()['@id']
    print(f"Loading records from {rs_id} ...")
    try:
        recs = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(recs)
        print(f"  {len(df)} rows, columns: {list(df.columns)}")
        df_map[rs_id] = df
    except Exception as e:
        print(f"  Failed: {e}")

# Select a main record set for exploration: the largest populated one
primary_rs_id = None
max_rows = 0
for rs_id, df in df_map.items():
    if df.shape[0] > max_rows:
        primary_rs_id = rs_id
        max_rows = df.shape[0]
if primary_rs_id:
    df = df_map[primary_rs_id]
    print(f"\nPrimary record set for analysis: {primary_rs_id}")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())
else:
    print("No populated record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. Operations include removing outliers, transforming data distributions, and grouping by key attributes using `@id`s.

In [ ]:
import numpy as np

# Identify a numeric field for demonstration purposes
display_cols = df.columns if primary_rs_id else []
print("Numeric columns in the main data:")
numeric_candidates = []
for col in display_cols:
    if np.issubdtype(df[col].dtype, np.number):
        print(f"- {col}")
        numeric_candidates.append(col)
if not numeric_candidates:
    print("No numeric fields found.")
    numeric_field = None
else:
    numeric_field = numeric_candidates[0]

# Filtering and normalization (if numeric field exists)
if numeric_field:
    print(f"Will use: {numeric_field}")

    # Set a threshold (e.g., mean)
    threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}: {filtered_df.shape[0]}")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a (first non-numeric) field if present
    group_field = None
    for col in display_cols:
        if (col != numeric_field) and not np.issubdtype(df[col].dtype, np.number):
            group_field = col
            break
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric field available for filtering or normalization.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_rs_id and numeric_field:
    plt.figure(figsize=(10,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Scatter of normalized vs. raw if normalized field is present
    norm_col = f"{numeric_field}_normalized"
    if norm_col in filtered_df.columns:
        plt.figure(figsize=(7,5))
        sns.scatterplot(data=filtered_df, x=numeric_field, y=norm_col)
        plt.title(f'Scatter: {numeric_field} (raw) vs {norm_col}')
        plt.show()
else:
    print("No numeric field for visualization.")

## 6. Conclusion
This notebook demonstrated how to:
- Load a FAIR dataset defined by a Croissant schema using `mlcroissant`.
- Explore schema entities using the `@id` field for all references.
- Extract records and analyze primary record sets.
- Apply numerical filtering and normalization to fields, and visualize data distributions.

When working with FAIR datasets, referencing entities by their `@id` allows for robust schema-aware exploration and analysis.